<a href="https://colab.research.google.com/github/vaishnavireddy-03/plant-disease-classifier-final/blob/main/Plant_Disease_CNN_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import matplotlib.pyplot as plt
import numpy as np

In [2]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cpu


In [3]:
import kagglehub

path = kagglehub.dataset_download(
    "emmarex/plantdisease"
)

print(path)

Using Colab cache for faster access to the 'plantdisease' dataset.
/kaggle/input/plantdisease


In [4]:
transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

In [5]:
dataset_path = "/kaggle/input/plantdisease/PlantVillage"

dataset = datasets.ImageFolder(
    dataset_path,
    transform=transform
)

print(dataset.classes)
print(len(dataset.classes))
print("Images:", len(dataset))

['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']
15
Images: 20638


In [6]:
train_size = int(0.7 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_set, val_set, test_set = random_split(
    dataset,
    [train_size, val_size, test_size]
)

print(len(train_set))
print(len(val_set))
print(len(test_set))

14446
3095
3097


In [7]:
train_loader = DataLoader(
    train_set,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_set,
    batch_size=32
)

test_loader = DataLoader(
    test_set,
    batch_size=32
)

In [16]:
class PlantCNN(nn.Module):

    def __init__(self,num_classes):

        super().__init__()

        self.conv1 = nn.Conv2d(3,16,3)
        self.conv2 = nn.Conv2d(16,32,3)
        self.conv3 = nn.Conv2d(32,64,3)

        self.pool = nn.MaxPool2d(2,2)

        self.relu = nn.ReLU()

        self.fc1 = nn.Linear(12544,128)

        self.fc2 = nn.Linear(
            128,
            num_classes
        )

    def forward(self,x):

        x=self.pool(self.relu(self.conv1(x)))
        x=self.pool(self.relu(self.conv2(x)))
        x=self.pool(self.relu(self.conv3(x)))

        x=torch.flatten(x,1)

        x=self.relu(self.fc1(x))

        x=self.fc2(x)

        return x

In [17]:
model = PlantCNN(
    num_classes=len(dataset.classes)
).to(device)

print(model)

PlantCNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1))
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (relu): ReLU()
  (fc1): Linear(in_features=12544, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=15, bias=True)
)


In [18]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [29]:
train_losses=[]
val_losses=[]

train_accs=[]
val_accs=[]

epochs = 20

for epoch in range(epochs):

    model.train()

    running_loss=0
    correct=0
    total=0

    for images,labels in train_loader:

        images=images.to(device)
        labels=labels.to(device)

        optimizer.zero_grad()

        outputs=model(images)

        loss=criterion(outputs,labels)

        loss.backward()

        optimizer.step()

        running_loss+=loss.item()

        _,pred=torch.max(outputs,1)

        total+=labels.size(0)
        correct+=(pred==labels).sum().item()

    train_acc=100*correct/total

    train_losses.append(running_loss)
    train_accs.append(train_acc)

    model.eval()

    val_correct=0
    val_total=0

    with torch.no_grad():

        for images,labels in val_loader:

            images=images.to(device)
            labels=labels.to(device)

            outputs=model(images)

            _,pred=torch.max(outputs,1)

            val_total+=labels.size(0)
            val_correct+=(pred==labels).sum().item()

    val_acc=100*val_correct/val_total

    val_accs.append(val_acc)

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Val Acc: {val_acc:.2f}%"
    )

Epoch 1/20 | Train Acc: 67.68% | Val Acc: 83.13%
Epoch 2/20 | Train Acc: 84.38% | Val Acc: 87.46%
Epoch 3/20 | Train Acc: 89.53% | Val Acc: 88.89%
Epoch 4/20 | Train Acc: 92.44% | Val Acc: 92.83%
Epoch 5/20 | Train Acc: 93.52% | Val Acc: 90.50%
Epoch 6/20 | Train Acc: 94.34% | Val Acc: 93.44%
Epoch 7/20 | Train Acc: 95.38% | Val Acc: 93.63%
Epoch 8/20 | Train Acc: 96.05% | Val Acc: 93.99%
Epoch 9/20 | Train Acc: 96.19% | Val Acc: 92.18%
Epoch 10/20 | Train Acc: 96.96% | Val Acc: 93.28%
Epoch 11/20 | Train Acc: 97.06% | Val Acc: 95.28%
Epoch 12/20 | Train Acc: 97.08% | Val Acc: 93.05%
Epoch 13/20 | Train Acc: 97.63% | Val Acc: 94.41%
Epoch 14/20 | Train Acc: 97.63% | Val Acc: 92.99%
Epoch 15/20 | Train Acc: 97.67% | Val Acc: 95.12%
Epoch 16/20 | Train Acc: 98.07% | Val Acc: 94.35%
Epoch 17/20 | Train Acc: 97.71% | Val Acc: 94.41%
Epoch 18/20 | Train Acc: 98.06% | Val Acc: 94.51%
Epoch 19/20 | Train Acc: 98.59% | Val Acc: 91.66%
Epoch 20/20 | Train Acc: 98.01% | Val Acc: 95.28%


In [32]:
cm = confusion_matrix(
    all_labels,
    all_preds
)

print(cm)

print(
    classification_report(
        all_labels,
        all_preds
    )
)

[[  0   0   0   0   0   0   0   0   0   0   0 138   0   0  21]
 [  0   0   0   0   0   0   0   0   0   0   0 174   0   0  49]
 [  0   0   0   0   0   0   0   0   0   0   0 101   0   0  33]
 [  0   0   0   0   0   0   0   0   0   0   0  87   0   0  42]
 [  0   0   0   0   0   0   0   0   0   0   0  16   0   0   4]
 [  0   0   0   0   0   0   0   0   0   0   0 175   0   0 143]
 [  0   0   0   0   0   0   0   0   0   0   0 120   0   0  38]
 [  0   0   0   0   0   0   0   0   0   0   0 138   2   0 132]
 [  0   0   0   0   0   0   0   0   0   0   0  70   0   0  76]
 [  0   0   0   0   0   0   0   0   0   0   0 179   0   0  81]
 [  0   0   0   0   0   0   0   0   0   0   0 173   0   0  60]
 [  0   0   0   0   0   0   0   0   0   0   0 159   0   0  63]
 [  0   0   0   0   0   0   0   0   0   0   0 415   0   0  92]
 [  0   0   0   0   0   0   0   0   0   0   0  15   0   0  32]
 [  0   0   0   0   0   0   0   0   0   0   0 143   0   0 126]]
              precision    recall  f1-score   support


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [33]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

accuracy = accuracy_score(all_labels, all_preds)

precision = precision_score(
    all_labels,
    all_preds,
    average='weighted',
    zero_division=0
)

recall = recall_score(
    all_labels,
    all_preds,
    average='weighted',
    zero_division=0
)

f1 = f1_score(
    all_labels,
    all_preds,
    average='weighted',
    zero_division=0
)

print("Test Accuracy:", accuracy * 100)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

print("\nClassification Report:\n")
print(classification_report(
    all_labels,
    all_preds,
    zero_division=0
))

cm = confusion_matrix(all_labels, all_preds)

print("\nConfusion Matrix:\n")
print(cm)

Test Accuracy: 94.8337100419761
Precision: 0.9505643701806622
Recall: 0.948337100419761
F1 Score: 0.9484468561218555

Classification Report:

              precision    recall  f1-score   support

           0       0.95      0.92      0.94       159
           1       0.98      0.96      0.97       223
           2       0.98      0.98      0.98       134
           3       0.97      0.86      0.91       129
           4       0.90      0.95      0.93        20
           5       0.96      0.99      0.98       318
           6       0.92      0.85      0.89       158
           7       0.88      0.95      0.91       272
           8       0.98      0.87      0.92       146
           9       0.85      0.97      0.91       260
          10       0.93      0.96      0.95       233
          11       0.95      0.90      0.92       222
          12       1.00      0.97      0.98       507
          13       0.98      1.00      0.99        47
          14       0.99      0.98      0.99    

In [34]:
torch.save(model.state_dict(), "plantcnn.pth")

In [35]:
import os

print(
    "Model Size:",
    os.path.getsize("plantcnn.pth")/(1024*1024),
    "MB"
)

Model Size: 6.2267656326293945 MB


In [37]:
from google.colab import files
files.download("plantcnn.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

print(total_params)

1631279


In [28]:
print(model)

PlantCNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1))
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (relu): ReLU()
  (fc1): Linear(in_features=12544, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=15, bias=True)
)
